# 🇮🇳 BharatBuilds RL Training
**Training an LLM to be a human-first startup co-founder for Indian entrepreneurs**

Uses: `openenv-core` + `TRL` (GRPO) + `Unsloth` on `Qwen2.5-1.5B-Instruct`

The LLM learns to give empathetic, jargon-free, resource-aware guidance  
to Tier 2/3 city founders — without deciding for them.

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────
!pip install -q unsloth trl>=0.17.0 openenv-core>=0.2.3 httpx wandb
!pip install -q 'torch>=2.1' --index-url https://download.pytorch.org/whl/cu121

In [ ]:
# ── 2. Clone / install BharatBuildsEnv ──────────────────────
!git clone https://huggingface.co/spaces/YOUR_HF_USERNAME/BharatBuildsEnv /tmp/BharatBuildsEnv
import sys
sys.path.insert(0, '/tmp/BharatBuildsEnv')

In [ ]:
# ── 3. Imports ───────────────────────────────────────────────
import json, re, random
import torch
import wandb
from typing import List

from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset

# Import the environment directly (local mode, no server needed for training)
from environment import BharatBuildsEnv, BharatAction, FOUNDERS, IDEAS, PHASES
from verifiers import run_all_verifiers

print('Imports OK ✓')

In [ ]:
# ── 4. Load model with Unsloth ───────────────────────────────
MODEL_NAME = 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit'  # fast, fits in T4
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('Model loaded ✓')

In [ ]:
# ── 5. Prompt builder ────────────────────────────────────────
SYSTEM_PROMPT = """You are BharatBuilds AI — a human-first startup co-founder assistant for Indian entrepreneurs from Tier 2/3 cities.

Your role is to be their thinking partner, not their decision-maker.
RULES:
- Never use unexplained jargon (TAM, CAC, ARR, etc.) for low-literacy founders
- Never decide FOR the founder — always guide them to decide themselves
- Always recommend real, FREE or low-cost Indian resources
- Match your language to the founder's (Hinglish, Hindi, English)
- Give ONE concrete, doable task — not a list
- When founder is discouraged, be extra encouraging before pushing

Respond ONLY with valid JSON (no markdown, no explanation outside the JSON):
{
  "ai_response": "your warm, jargon-free, 2-3 sentence response to the founder",
  "suggested_task": "one specific, doable action for them to take today (under 30 words)",
  "task_rationale": "one sentence: why this task moves them forward",
  "used_jargon": false,
  "made_decision_for_human": false,
  "resource_recommended": "name one specific free resource or empty string",
  "emotional_tone": "encouraging"
}"""

def build_prompt(obs: dict) -> str:
    schemes  = ', '.join(obs.get('available_schemes', [])[:2])
    tools    = ', '.join(obs.get('available_tools', [])[:2])
    communities = ', '.join(obs.get('available_communities', [])[:1])
    user_msg = f"""FOUNDER: {obs['founder_name']} | {obs['founder_location']} | {obs['founder_tier']}
LANGUAGE: {obs['founder_language']} | DIGITAL LITERACY: {obs['founder_digital_literacy']:.1f}/1.0
CAPITAL: ₹{obs['founder_capital_inr']:,.0f} | MOOD: {obs['founder_emotional_state']}

IDEA: {obs['idea_description']}

CURRENT PHASE: {obs['phase']} (Phase {obs['phase_number']+1}/7)
PHASE GOAL: {obs['phase_goal']}

PROGRESS:
- Validation interviews: {obs['validation_interviews_done']}/5
- MVP shipped: {obs['mvp_shipped']}
- First customer: {obs['first_customer']}
- Dropout risk: {obs['dropout_risk']:.0%}

FREE RESOURCES AVAILABLE:
- Schemes: {schemes}
- Tools: {tools}
- Community: {communities}

What do you say to this founder right now?"""
    return user_msg

def apply_chat_template(obs: dict) -> str:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': build_prompt(obs)},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print('Prompt builder OK ✓')

In [ ]:
# ── 6. Action parser ─────────────────────────────────────────
DEFAULT_ACTION = {
    'ai_response': '',
    'suggested_task': '',
    'task_rationale': '',
    'used_jargon': False,
    'made_decision_for_human': False,
    'resource_recommended': '',
    'emotional_tone': 'encouraging',
}

def parse_action(text: str) -> dict:
    """Parse model output into BharatAction dict. Robust to partial JSON."""
    text = text.strip()
    # Try direct JSON parse
    try:
        return {**DEFAULT_ACTION, **json.loads(text)}
    except Exception:
        pass
    # Try extracting JSON block
    m = re.search(r'\{.*\}', text, re.DOTALL)
    if m:
        try:
            return {**DEFAULT_ACTION, **json.loads(m.group())}
        except Exception:
            pass
    # Fallback — return default with raw text as ai_response
    return {**DEFAULT_ACTION, 'ai_response': text[:500]}

print('Parser OK ✓')

In [ ]:
# ── 7. Reward function for GRPO ──────────────────────────────
# GRPO calls reward_fn(prompts, completions, **kwargs) -> List[float]
# We run one env step per completion and return the scalar reward.

# We store env instances keyed by prompt hash so we can step them.
# For GRPO, each prompt corresponds to an env observation.
_env_cache = {}

def reward_fn(prompts: List[str], completions: List[str], obs_list=None, **kwargs) -> List[float]:
    rewards = []
    for i, (prompt, completion) in enumerate(zip(prompts, completions)):
        action_dict = parse_action(completion)
        action = BharatAction(**{k: action_dict.get(k, v) for k, v in DEFAULT_ACTION.items()})
        
        # Get the observation for this prompt from kwargs
        obs = obs_list[i] if obs_list else {}
        
        # Verifier scores (independent of env simulation)
        v_state = {
            'phase': obs.get('phase', 'IDEA_ARTICULATION'),
            'founder': {
                'digital_literacy': obs.get('founder_digital_literacy', 0.5),
                'capital_inr': obs.get('founder_capital_inr', 10000),
                'language': obs.get('founder_language', 'english'),
            },
            'engagement': {
                'dropout_risk': obs.get('dropout_risk', 0.0),
                'tasks_completed': obs.get('tasks_completed', 0),
            },
        }
        vresult = run_all_verifiers(action.dict(), v_state)
        
        # JSON format reward: did the model produce valid JSON?
        try:
            json.loads(completion.strip())
            format_reward = 5.0
        except Exception:
            format_reward = -10.0
        
        # Combine: verifier composite + format reward
        # (Env simulation reward is handled during rollout collection for full episodes)
        total = vresult.total + format_reward
        
        # Clip to reasonable range
        total = max(-50.0, min(60.0, total))
        rewards.append(total)
    
    return rewards

print('Reward function OK ✓')

In [ ]:
# ── 8. Build training dataset ────────────────────────────────
# We sample random (founder, phase) combinations and build prompts.
# This gives GRPO diverse episodes to learn from.

def generate_training_samples(n: int = 500) -> Dataset:
    samples = []
    env = BharatBuildsEnv()
    
    for i in range(n):
        obs = env.reset(seed=i)
        obs_dict = obs.dict()
        
        # Sometimes fast-forward to a random phase for diversity
        target_phase = random.randint(0, 6)
        for _ in range(target_phase * 3):
            if obs_dict.get('done'): break
            # Random benign action to advance state
            dummy = BharatAction(
                ai_response='Let us take one small step forward.',
                suggested_task='Call one person you know about this idea.',
                task_rationale='Real feedback beats assumptions.',
                used_jargon=False,
                made_decision_for_human=False,
                resource_recommended='WhatsApp Business free',
                emotional_tone='encouraging'
            )
            obs = env.step(dummy)
            obs_dict = obs.dict()
        
        prompt = apply_chat_template(obs_dict)
        samples.append({'prompt': prompt, 'obs': json.dumps(obs_dict)})
    
    return Dataset.from_list(samples)

print('Generating 500 training samples...')
train_dataset = generate_training_samples(500)
print(f'Dataset size: {len(train_dataset)} ✓')
print('Sample prompt (first 400 chars):')
print(train_dataset[0]['prompt'][:400])

In [ ]:
# ── 9. Wrap reward for GRPO (receives prompts + completions) ──
import json as _json

def grpo_reward_fn(prompts, completions, obs=None, **kwargs):
    """GRPO-compatible reward. Extracts obs from dataset extras."""
    obs_list = []
    if obs is not None:
        for o in obs:
            try:
                obs_list.append(_json.loads(o) if isinstance(o, str) else o)
            except Exception:
                obs_list.append({})
    else:
        obs_list = [{} for _ in prompts]
    return reward_fn(prompts, completions, obs_list=obs_list)

In [ ]:
# ── 10. GRPO Training config ─────────────────────────────────
wandb.init(project='bharat-builds-rl', name='grpo-qwen2.5-1.5b')

grpo_config = GRPOConfig(
    output_dir='./bharat-builds-grpo',
    
    # Core GRPO settings
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,         # rollouts per prompt
    
    # Generation settings
    max_prompt_length=1024,
    max_completion_length=512,
    temperature=0.9,
    
    # Optimization
    learning_rate=5e-6,
    warmup_ratio=0.1,
    optim='adamw_8bit',
    
    # GRPO-specific
    beta=0.04,                  # KL penalty coefficient
    epsilon=0.2,                # clip ratio
    
    # Logging
    logging_steps=5,
    save_steps=50,
    report_to='wandb',
    
    # Efficiency
    fp16=True,
    dataloader_num_workers=0,
    remove_unused_columns=False,
)

print('GRPO config OK ✓')

In [ ]:
# ── 11. Train ─────────────────────────────────────────────────
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[grpo_reward_fn],
    args=grpo_config,
    train_dataset=train_dataset,
)

print('Starting GRPO training...')
trainer.train()
print('Training complete ✓')

In [ ]:
# ── 12. Save model correctly (Unsloth method) ─────────────────
model.save_pretrained_merged(
    'bharat-builds-final',
    tokenizer,
    save_method='merged_16bit',
)
print('Model saved ✓')

# Push to HF Hub
# model.push_to_hub('YOUR_HF_USERNAME/bharat-builds-rl')
# tokenizer.push_to_hub('YOUR_HF_USERNAME/bharat-builds-rl')

In [ ]:
# ── 13. Before/After Evaluation ───────────────────────────────
# Run the trained model on 20 episodes and compare reward to a random baseline

import numpy as np
import matplotlib.pyplot as plt

def run_episode(model, tokenizer, founder_name=None, max_steps=15):
    """Run one full episode with the given model. Returns (rewards, phases_reached)."""
    env = BharatBuildsEnv()
    obs = env.reset(founder_name=founder_name)
    obs_dict = obs.dict()
    
    episode_rewards = []
    phases_reached = set([obs_dict['phase']])
    
    FastLanguageModel.for_inference(model)
    
    for step in range(max_steps):
        if obs_dict.get('done'):
            break
        
        prompt = apply_chat_template(obs_dict)
        inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=300,
                temperature=0.7,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        
        completion = tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True
        )
        action_dict = parse_action(completion)
        action = BharatAction(**{k: action_dict.get(k, v) for k, v in DEFAULT_ACTION.items()})
        
        obs = env.step(action)
        obs_dict = obs.dict()
        episode_rewards.append(obs_dict['reward'])
        phases_reached.add(obs_dict['phase'])
    
    return episode_rewards, phases_reached, env.state.cumulative_reward

def run_random_baseline(max_steps=15):
    """Random policy baseline — outputs garbage JSON."""
    env = BharatBuildsEnv()
    obs = env.reset()
    rewards = []
    for _ in range(max_steps):
        if obs.dict().get('done'):
            break
        action = BharatAction(
            ai_response='You should definitely raise venture capital funding right away.',
            suggested_task='Pitch to 50 VCs this week and build a scalable SaaS platform.',
            task_rationale='ARR and CAC optimization drives PMF achievement.',
            used_jargon=True,
            made_decision_for_human=True,
            resource_recommended='Hire a dev team at 5 lakh per month',
            emotional_tone='neutral'
        )
        obs = env.step(action)
        rewards.append(obs.dict()['reward'])
    return rewards, env.state.cumulative_reward

print('Running evaluation...')
N_EVAL = 10
trained_totals, baseline_totals = [], []

for i in range(N_EVAL):
    founder = random.choice(['Priya','Ravi','Meera','Suresh','Anjali'])
    _, _, cum = run_episode(model, tokenizer, founder_name=founder)
    trained_totals.append(cum)
    _, bcum = run_random_baseline()
    baseline_totals.append(bcum)

print(f'Trained model avg cumulative reward: {np.mean(trained_totals):.1f} ± {np.std(trained_totals):.1f}')
print(f'Random baseline avg cumulative reward: {np.mean(baseline_totals):.1f} ± {np.std(baseline_totals):.1f}')

In [ ]:
# ── 14. Plot results ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Cumulative reward comparison
ax = axes[0]
ax.bar(['Random Baseline', 'GRPO Trained'],
       [np.mean(baseline_totals), np.mean(trained_totals)],
       yerr=[np.std(baseline_totals), np.std(trained_totals)],
       color=['#FF6B6B', '#4CAF50'], capsize=6, width=0.5)
ax.set_title('Cumulative Reward per Episode', fontsize=13, fontweight='bold')
ax.set_ylabel('Cumulative Reward')
ax.set_xlabel('Policy')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
for i, v in enumerate([np.mean(baseline_totals), np.mean(trained_totals)]):
    ax.text(i, v + 2, f'{v:.1f}', ha='center', fontweight='bold')

# Plot 2: Per-step reward across episodes (trained model)
ax2 = axes[1]
# Show reward progression over training steps from wandb-style logs (mock if needed)
# In real run, replace with: history = wandb.run.history(); ax2.plot(history['reward'])
x = np.arange(N_EVAL)
ax2.plot(x, trained_totals, 'o-', color='#4CAF50', label='Trained', linewidth=2)
ax2.plot(x, baseline_totals, 's--', color='#FF6B6B', label='Baseline', linewidth=2)
ax2.set_title('Cumulative Reward per Eval Episode', fontsize=13, fontweight='bold')
ax2.set_xlabel('Evaluation Episode')
ax2.set_ylabel('Cumulative Reward')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('BharatBuilds RL Training Results', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('bharat_builds_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plots saved to bharat_builds_results.png ✓')

In [ ]:
# ── 15. Qualitative before/after comparison ───────────────────
TEST_OBS = {
    'founder_name': 'Priya',
    'founder_location': 'Jhansi, UP',
    'founder_tier': 'tier3',
    'founder_language': 'hinglish',
    'founder_domain': 'handicrafts',
    'founder_digital_literacy': 0.3,
    'founder_capital_inr': 5000,
    'founder_emotional_state': 'excited',
    'idea_description': 'I want to help women in my village sell their handmade things online',
    'phase': 'IDEA_ARTICULATION',
    'phase_number': 0,
    'phase_goal': 'Help founder turn a vague idea into a clear problem statement.',
    'validation_interviews_done': 0,
    'mvp_shipped': False,
    'first_customer': False,
    'dropout_risk': 0.0,
    'tasks_completed': 0,
    'tasks_ignored': 0,
    'available_schemes': ['MUDRA Shishu Loan up to 50k free', 'WEP Women Entrepreneurship Platform free'],
    'available_tools': ['Meesho seller zero investment', 'WhatsApp Catalog free'],
    'available_communities': ['DIC District Industries Centre'],
}

prompt = apply_chat_template(TEST_OBS)
inputs = tokenizer(prompt, return_tensors='pt').to('cuda')

FastLanguageModel.for_inference(model)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=350, temperature=0.7, do_sample=True)

response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
parsed = parse_action(response)

print('=' * 60)
print('TRAINED MODEL RESPONSE FOR PRIYA (Jhansi, handicrafts)')
print('=' * 60)
print(f"AI Response: {parsed['ai_response']}")
print(f"Suggested Task: {parsed['suggested_task']}")
print(f"Resource: {parsed['resource_recommended']}")
print(f"Used Jargon: {parsed['used_jargon']} | Decided for human: {parsed['made_decision_for_human']}")
print()
print('BASELINE (bad policy) would say:')
print('"You should raise venture capital. Build a scalable SaaS with PMF."')
print('(jargon=True, made_decision_for_human=True → heavy penalty)')